# Exercise: Zero-Shot Forecasting with a Foundation Model

Chronos-2 is a pretrained foundation model that can forecast any time series without training. Run it zero-shot on the subscriber data and compare to ARIMA. Build a comparison table of backtest metrics.

In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape

HORIZON = 12

In [ ]:
def run_chronos_zero_shot(train, horizon=12):
    """
    Run Chronos-2 zero-shot forecast. Falls back to ARIMA if unavailable.
    
    Parameters
    ----------
    train : darts.TimeSeries
    horizon : int
        Number of steps to forecast.
    
    Returns
    -------
    darts.TimeSeries
        Forecast with prediction intervals.
    """
    try:
        from darts.models import Chronos2Model
        model = Chronos2Model(model_name=...)
        return model.predict(horizon, series=train, num_samples=...)
    except ImportError:
        fallback = DartsARIMA(p=..., d=..., q=...)
        fallback.fit(train)
        return fallback.predict(horizon)

chronos_fc = run_chronos_zero_shot(train, horizon=HORIZON)

## Task 1: Run Chronos-2 Zero-Shot

Load `Chronos2Model` with the `"amazon/chronos-bolt-small"` checkpoint. Call `predict` with `num_samples=100`. No `fit()` needed. If Chronos-2 is not installed, fall back to Darts ARIMA.

In [ ]:
arima = DartsARIMA(p=2, d=1, q=1)
arima.fit(train)
arima_fc = arima.predict(HORIZON)

def backtest_model(model, series, start=0.75):
    """
    Backtest a model using historical forecasts.
    
    Parameters
    ----------
    model : darts.models.*
    series : darts.TimeSeries
    start : float
        Fraction of series to use before starting backtest.
    
    Returns
    -------
    dict
        Keys: mae, rmse, mape
    """
    preds = model.historical_forecasts(
        series,
        forecast_horizon=...,
        stride=...,
        retrain=...,
        start=...
    )
    return {
        "mae": mae(series, preds),
        "rmse": rmse(series, preds),
        "mape": mape(series, preds)
    }

arima_metrics = backtest_model(arima, TimeSeries.from_series(subscribers))

def build_comparison_table(models):
    """
    Build a comparison DataFrame from model metrics.
    
    Parameters
    ----------
    models : dict
        {model_name: {mae, rmse, mape}}
    
    Returns
    -------
    pd.DataFrame
    """
    rows = []
    for name, metrics in models.items():
        rows.append({
            "Model": name,
            "MAE": metrics["mae"],
            "RMSE": metrics["rmse"],
            "MAPE": metrics["mape"]
        })
    return pd.DataFrame(rows).set_index("Model")

metrics = {"ARIMA(2,1,1)": arima_metrics}
table = build_comparison_table(metrics)
print(table)

In [ ]:
def build_comparison_table(models):
    """
    Build a comparison DataFrame from model metrics.
    
    Parameters
    ----------
    models : dict
        {model_name: {mae, rmse, mape}}
    
    Returns
    -------
    pd.DataFrame
    """
    rows = []
    for name, metrics in models.items():
        rows.append({
            "Model": name,
            "MAE": metrics["mae"],
            "RMSE": metrics["rmse"],
            "MAPE": metrics["mape"]
        })
    return pd.DataFrame(rows).set_index("Model")

metrics = {"ARIMA(2,1,1)": arima_metrics}
table = build_comparison_table(metrics)
print(table)